Carga de Base de Datos


In [2]:
import pandas as pd 
import numpy as np

df = pd.read_csv('../Base_de_datos.csv')

df_preparado = pd.get_dummies(df, drop_first=True)

df_preparado.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Columns: 5018 entries, signup_month to plan_Premium
dtypes: bool(5007), float64(3), int64(8)
memory usage: 24.3 MB


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

x = df_preparado.drop(columns=['churn'])
y = df_preparado['churn']

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

modelo_log = LogisticRegression(max_iter=1000)
modelo_log.fit(X_train, y_train)

modelo_arbol = DecisionTreeClassifier(random_state=42)
modelo_arbol.fit(X_train, y_train)

print("Modelos entrenados con exito!")

c:\Users\Julian\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Modelos entrenados con exito!


In [4]:
import joblib
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

pred_log = modelo_log.predict(X_test)
pred_arbol = modelo_arbol.predict(X_test)

acc_log = accuracy_score(y_test, pred_log)
prec_log = precision_score(y_test, pred_log, zero_division=0)
rec_log = recall_score(y_test, pred_log, zero_division=0)
f1_log = f1_score(y_test, pred_log, zero_division=0)

acc_arbol = accuracy_score(y_test, pred_arbol)
prec_arbol = precision_score(y_test, pred_arbol, zero_division=0)
rec_arbol = recall_score(y_test, pred_arbol, zero_division=0)
f1_arbol = f1_score(y_test, pred_arbol, zero_division=0)

print(" EVALUACIÓN Y COMPARACIÓN ROBUSTA DE RENDIMIENTOS")
print("-" * 60)
print(f"| Métrica    | Regresión Logística | Árbol de Decisión |")
print("-" * 60)
print(f"| Accuracy   | {acc_log*100:18.2f}% | {acc_arbol*100:16.2f}% |")
print(f"| Precision  | {prec_log*100:18.2f}% | {prec_arbol*100:16.2f}% |")
print(f"| Recall     | {rec_log*100:18.2f}% | {rec_arbol*100:16.2f}% |")
print(f"| F1-Score   | {f1_log*100:18.2f}% | {f1_arbol*100:16.2f}% |")
print("-" * 60)

print("\n CONCLUSIÓN Y ANÁLISIS DE SELECCIÓN:")
if acc_log > acc_arbol:
    print("El modelo seleccionado es la Regresión Logística debido a su rendimiento superior general.")
    print("Presenta un balance óptimo entre la Exactitud (Accuracy) y la capacidad de detectar")
    print("clientes en riesgo de abandono (Recall), minimizando los falsos negativos operativos.")
    modelo_ganador = modelo_log
else:
    print("El modelo seleccionado es el Árbol de Decisión debido a su rendimiento superior.")
    modelo_ganador = modelo_arbol

print("\n Exportando artefactos reales para la API de producción...")
if not os.path.exists('src'):
    os.makedirs('src')

joblib.dump(modelo_ganador, 'src/modelo_churn.pkl')
joblib.dump(list(x.columns), 'src/columnas_modelo.pkl')
print(" ¡Exito! Archivos 'src/modelo_churn.pkl' y 'src/columnas_modelo.pkl' generados.")


 EVALUACIÓN Y COMPARACIÓN ROBUSTA DE RENDIMIENTOS
------------------------------------------------------------
| Métrica    | Regresión Logística | Árbol de Decisión |
------------------------------------------------------------
| Accuracy   |              74.80% |            68.60% |
| Precision  |              43.66% |            31.41% |
| Recall     |              12.76% |            24.69% |
| F1-Score   |              19.75% |            27.65% |
------------------------------------------------------------

 CONCLUSIÓN Y ANÁLISIS DE SELECCIÓN:
El modelo seleccionado es la Regresión Logística debido a su rendimiento superior general.
Presenta un balance óptimo entre la Exactitud (Accuracy) y la capacidad de detectar
clientes en riesgo de abandono (Recall), minimizando los falsos negativos operativos.

 Exportando artefactos reales para la API de producción...
 ¡Exito! Archivos 'src/modelo_churn.pkl' y 'src/columnas_modelo.pkl' generados.
